In [0]:
%fs
ls /Volumes/workspace/ecommerce/ecommerce_data

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.ecommerce.gold_events")

df_clean = df.dropna()

# StringIndexer for categorical column 'event_type'
from pyspark.ml.feature import StringIndexer, VectorAssembler
indexer = StringIndexer(inputCol="event_type", outputCol="event_type_index")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# Assemble features: total_sales, event_count, avg_price, event_type_index
assembler = VectorAssembler(
    inputCols=["total_sales", "event_count", "avg_price", "event_type_index"],
    outputCol="features"
)
df_prepped = assembler.transform(df_indexed)

display(df_prepped)

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from mlflow.models import infer_signature

# Convert Spark DataFrame to Pandas
pdf = df_prepped.select("total_sales", "event_count", "avg_price", "event_type_index", "features").toPandas()

# Features and target
X = pdf[["event_count", "avg_price", "event_type_index"]].astype(float)
y = pdf["total_sales"].astype(float)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression
with mlflow.start_run(run_name="linear_regression_gold_events"):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    signature = infer_signature(X_train, model.predict(X_train))
    input_example = X_train.head(5)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="gold_events_total_sales_regression"
    )
    print("Linear Regression RMSE:", rmse)
    print("Linear Regression R2:", r2)

# Ridge Regression
with mlflow.start_run(run_name="ridge_regression_gold_events"):
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    signature = infer_signature(X_train, model.predict(X_train))
    input_example = X_train.head(5)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="gold_events_total_sales_regression"
    )
    print("Ridge Regression RMSE:", rmse)
    print("Ridge Regression R2:", r2)
